# demo_rl16

> Phase 16 (`specs/multiagent/phase16/`): more training seeds for the incentive result. Reads the evaluation rows of the nine runs (`individual` s1-s4, `indpen` s1-s5) and the warm start, runs `RL16_PLAN`'s exact tests with the training run as the unit, and writes the report and figure.

Artifacts (gitignored): `results/multiagent/rl/quick_p16/` (synthetic) and `results/multiagent/rl/eval_p16/` (`report.md`, `rl16.png`, `summary.json`). Inputs for the full analysis: `results/multiagent/rl/eval/runs.csv` (warm start, `individual_s1`), `results/multiagent/rl/eval_p13/runs.csv` (`indpen_s1`, `indpen_s2`) and `results/multiagent/rl/colab/eval_p16/runs.csv` (the six new runs, from the Colab batch's zip, unzipped into `results/multiagent/rl/colab/`).

- **Quick** (normal tests): synthetic runs with known effects, end to end.
- **Full** (`--flags train`): only after `RL16_PLAN` is frozen and the Colab batch has finished.

In [ ]:
import json
import subprocess
import sys
from pathlib import Path

from explRL.bench_grid import readCSV
from explRL.coop_grid import coopHash
from explRL.rl16_grid import RL16_PLAN
from explRL.rl16_report import figRL16, rl16Summaries, syntheticRows, writeRL16
from explRL.train import demo_artifact_dir

RL = demo_artifact_dir(None, 'multiagent/rl')
ROOT = RL.parents[2]


def gitCommit():
    try:
        return subprocess.run(['git', 'rev-parse', '--short', 'HEAD'], capture_output=True, text=True).stdout.strip() or 'unknown'
    except OSError:
        return 'unknown'


def gpuHours(runs):
    "Training GPU hours of the new runs, from their per-step rollout and update seconds."
    t = 0.0
    for p in sorted(Path(runs).glob('*/metrics.jsonl')):
        t += sum(json.loads(x).get('rolloutSeconds', 0) + json.loads(x).get('updateSeconds', 0) for x in open(p) if x.strip())
    return round(t / 3600, 2)


def analyse(mode, rows12, rows13, rows16, out, nboot=2000, hours='n/a'):
    out.mkdir(parents=True, exist_ok=True)
    S = rl16Summaries(rows12, rows13, rows16, nboot)
    meta = dict(mode=mode, planHash=coopHash(RL16_PLAN), commit=gitCommit(), gpuHours=hours)
    writeRL16(out / 'report.md', S, meta)
    figRL16(out, S)
    (out / 'summary.json').write_text(json.dumps(dict(meta, **S), indent=1, sort_keys=True, default=str) + '\n')
    return S

## Quick: synthetic runs

In [ ]:
qo = RL / 'quick_p16'
S = analyse('quick', *syntheticRows(), qo, nboot=300)
assert all((qo / f).stat().st_size > 0 for f in ('report.md', 'rl16.png', 'summary.json'))
assert S['claims'] == dict(sharingTested=True, scoreContrastTested=True, beatsImitation=True, driftReplicated=True)
S0 = analyse('quick', *syntheticRows(seed=3, share=(0.9, 0.9), solo=(0.03, 0.03), welfare=(0.27, 0.27)), qo, nboot=300)
assert not any(S0['claims'].values())
print(open(qo / 'report.md').read()[:1200])

## Full (`--flags train`): the nine runs and the warm start

In [ ]:
#| train
p16 = RL / 'colab' / 'eval_p16' / 'runs.csv'
if not p16.exists():
    print('skipped: unzip the Phase 16 Colab batch (explRL_rl16.zip) into results/multiagent/rl/colab/ first')
else:
    assert RL16_PLAN['frozen'], 'the analysis uses the frozen RL16_PLAN'
    S = analyse('full', readCSV(RL / 'eval' / 'runs.csv'), readCSV(RL / 'eval_p13' / 'runs.csv'), readCSV(p16), RL / 'eval_p16',
                hours=gpuHours(RL / 'colab' / 'runs_p16'))
    subprocess.run([sys.executable, str(ROOT / 'tools' / 'incentive_figure.py'), str(ROOT)], check=True)
    print(open(RL / 'eval_p16' / 'report.md').read())